# market_neural_net — Colab training launcher

Reusable GPU launcher shell per the project README (§2.1: local CPU handles data/backtesting, this notebook handles GPU-hungry training).

Code comes straight from GitHub (https://github.com/yuvidewan/market_neural_net) — no manual zip/upload step needed for that anymore. Only the curated **data** needs a one-time Drive upload (it stays out of git on purpose — see `.gitignore`):
- Run `python -m scripts.package_for_colab --skip-data` locally if you ever want the old zip-based code path instead (e.g. offline, or before you've pushed a change) — not needed for normal use now.
- Upload `experiments/colab_data_bundle.zip` (~350MB, produced by the same script) to `My Drive/market_neural_net/colab_data_bundle.zip` — once, and again only after a real re-ingest of the curated dataset.

**Known gap, read before a long run:** none of the training scripts here checkpoint/resume mid-run yet. Fine for the LSTM baseline and the TCN SSL run (should both be well under an hour on a T4), but MUST be added before the two-axis transformer's full pretraining — that one genuinely runs for hours, and Colab sessions disconnect.

## 1. Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU assigned — Runtime > Change runtime type > GPU, then re-run this cell.')

## 2. Clone the repo

In [ ]:
REPO_URL = 'https://github.com/yuvidewan/market_neural_net.git'
PROJECT_DIR = '/content/market_neural_net'

import os
if os.path.exists(PROJECT_DIR):
    %cd $PROJECT_DIR
    !git pull
else:
    !git clone $REPO_URL $PROJECT_DIR
    %cd $PROJECT_DIR

## 3. Mount Drive and unpack the curated data
Data stays out of git (see `.gitignore`) — this is the one thing that still needs a manual Drive upload.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR = '/content/drive/My Drive/market_neural_net'  # change if you uploaded elsewhere

In [ ]:
import zipfile, os

data_zip = f'{DRIVE_PROJECT_DIR}/colab_data_bundle.zip'
assert os.path.exists(data_zip), (
    f'missing {data_zip} — run `python -m scripts.package_for_colab` locally and upload '
    f'experiments/colab_data_bundle.zip to that Drive path first'
)
with zipfile.ZipFile(data_zip) as zf:
    zf.extractall(PROJECT_DIR)
print('data extracted into', PROJECT_DIR)

## 4. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 5. Sanity checks: tests, then tiny smoke-test runs
Per README §2.1 rule 1 — always verify the tiny config before trusting a real run on a new machine.

In [ ]:
!python -m pytest tests/ -q

In [ ]:
# M2 tiny smoke test — should finish in well under a minute even on CPU.
!python -u -m scripts.train_lstm_baseline \
  --n-symbols 5 --seq-len 20 --hidden-size 16 --num-layers 1 --epochs 1 \
  --test-years 2024 --out-dir experiments/lstm_baseline_smoketest

In [ ]:
# M3 tiny smoke test (TCN + quantile regression) — should also finish in under a minute.
!python -u -m scripts.train_ssl_quantile \
  --n-symbols 5 --seq-len 20 --channels 16 --epochs 1 \
  --test-years 2024 --out-dir experiments/ssl_quantile_smoketest

## 6. M2 real run: LSTM baseline
Full README-spec baseline: 40 liquid symbols, seq_len=120, hidden=128, 2-layer LSTM, 4 walk-forward folds. On a T4 this should be well under the local-CPU runtime (a local CPU run of this exact config took 6.5 hours). Already run once locally — rerun here mainly to confirm parity, or after a code change.

`-u` keeps stdout unbuffered so progress prints as it happens instead of only at the end.

In [ ]:
!python -u -m scripts.train_lstm_baseline \
  --n-symbols 40 --seq-len 120 --hidden-size 128 --num-layers 2 --epochs 6 \
  --test-years 2022 2023 2024 2025 \
  --out-dir experiments/lstm_baseline

## 7. M3 real run: TCN + self-supervised quantile regression
**This is the one that actually needs the GPU** — the M3 gate (README): out-of-sample cross-sectional rank IC > 0.02, stable sign across all 4 walk-forward folds. TCN convolutions parallelize across time (unlike the LSTM's sequential recurrence), so this should scale to a meaningfully larger universe than M2's 40 symbols without another multi-hour surprise — 200 liquid names here, big enough that the daily cross-sectional rank correlation is measuring something statistically real rather than noise from a handful of names.

The report includes the M3 gate PASS/FAIL verdict directly — read the per-fold breakdown either way, a single overall number hides whether one fold is carrying the result.

In [ ]:
!python -u -m scripts.train_ssl_quantile \
  --n-symbols 200 --seq-len 120 --channels 64 --epochs 8 \
  --test-years 2022 2023 2024 2025 \
  --out-dir experiments/ssl_quantile_tcn

## 8. Sync results back to Drive
Results/checkpoints are run output, not source — they stay out of git and go to Drive instead, so they survive the Colab VM being recycled.

In [ ]:
import shutil
dest = f'{DRIVE_PROJECT_DIR}/experiments_from_colab'
shutil.copytree('experiments', dest, dirs_exist_ok=True)
print('synced experiments/ ->', dest)